# Lab 1C: Build Gold project context and milestone features

Run **Lab 1A: Bronze** and **Lab 1B: Silver** first. This notebook reads Silver tables and creates two Gold Delta tables in **seer_livelabs_20260922.seer_gold**.

Task 1 uses Spark SQL with named common table expressions (CTEs) to build project context. Task 2 uses the PySpark DataFrame API and `pyspark.sql.functions` to build milestone features. Both preserve native Spark table references and write Delta tables without Python UDFs, pandas conversions, or support-library imports.

Each task recreates only its named Gold table without a backup. Source files, schemas and volumes remain unchanged. Run **Lab 2** afterward to refresh predictions. Do not run concurrent writers.

## Rerunning Lab 1C

No manual deletion is required. The two write cells drop and recreate `project_context` and `milestone_features` without backups. Silver inputs, registered models, PDFs, volumes and the existing prediction table remain in place. A failed write after a drop may leave its target absent.

With current Silver inputs, start at setup and run both Gold tasks in order. If upstream data changed, rebuild the affected Bronze/Silver layers first. Run **Lab 2** afterward to replace predictions with scores for the refreshed Gold data; old predictions are not automatically removed or refreshed by Lab 1C. Resolve failed writes before scoring and avoid concurrent writers. Refresh the catalog and inspect outputs and lineage.

Lab 2 retains experiment history and creates a new model version on a complete run. No prior experiment/model deletion is needed. See the **Reruns and recovery** section of the repository's root `README.md`.


In [ ]:
import os, json
from pathlib import Path
from datetime import datetime, timezone
from pyspark.sql import functions as F

# All three layers use the workshop standard catalog.
config_path = "/Workspace/Shared/seer-aidp-livelabs/config/workshop.json"
RUN_ID = os.environ.get("SEER_RUN_ID", "interactive")
utilities = globals().get("oidlUtils")
if utilities is not None:
    config_path = utilities.parameters.getParameter("config_path", config_path)
    RUN_ID = utilities.parameters.getParameter("run_id", RUN_ID)
config = json.loads(Path(config_path).read_text())
SILVER = f"{config['catalog']}.{config['schemas']['silver']}"
GOLD = f"{config['catalog']}.{config['schemas']['gold']}"
keys = ["project_id", "asset_id"]

# Create the gold schema if it does not already exist.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}")


## Task 1: Build project context with Spark SQL

Use `spark.sql()` with named CTEs to aggregate purchase orders, suppliers, and inspections before joining them to project assets and the selected context milestone. This avoids multiplying costs across one-to-many relationships. Preserve the oldest source timestamp separately from this run's refresh timestamp. Named SQL parameters supply the refresh time and run ID without inserting their values into the query text.


In [ ]:
# Use named SQL steps (CTEs) to aggregate before joining, avoiding multiplied costs.
context = spark.sql(f"""
WITH orders_by_asset AS (
    SELECT project_id, asset_id,
           SUM(amount_cents) AS committed_cost_cents,
           COUNT(*) AS purchase_order_count,
           CONCAT_WS(' | ', SORT_ARRAY(COLLECT_SET(po_status))) AS purchase_order_status
    FROM {SILVER}.purchase_orders
    GROUP BY project_id, asset_id
),
-- Count each supplier only once per project asset.
distinct_suppliers AS (
    SELECT DISTINCT project_id, asset_id, supplier_id
    FROM {SILVER}.purchase_orders
),
suppliers_by_asset AS (
    SELECT o.project_id, o.asset_id,
           CONCAT_WS(' | ', SORT_ARRAY(COLLECT_SET(s.canonical_supplier_name))) AS supplier_names,
           MAX(CASE WHEN o.supplier_id IS NULL OR s.qualification_status <> 'APPROVED'
                    THEN 1 ELSE 0 END) AS supplier_needs_review
    FROM distinct_suppliers o
    LEFT JOIN {SILVER}.suppliers s ON o.supplier_id = s.supplier_id
    GROUP BY o.project_id, o.asset_id
),
-- Rank inspection outcomes: FAIL takes priority over REQUEST_INFO, then PASS.
inspection_totals AS (
    SELECT project_id, asset_id, COUNT(*) AS inspection_count,
           MAX(CASE WHEN result_status = 'FAIL' THEN 2
                    WHEN result_status = 'REQUEST_INFO' THEN 1
                    ELSE 0 END) AS inspection_rank
    FROM {SILVER}.inspections
    GROUP BY project_id, asset_id
),
inspections_by_asset AS (
    SELECT project_id, asset_id, inspection_count,
           CASE WHEN inspection_rank = 2 THEN 'FAIL'
                WHEN inspection_rank = 1 THEN 'REQUEST_INFO'
                ELSE 'PASS' END AS inspection_status
    FROM inspection_totals
),
-- These fixture milestones provide context; Task 2 selects predictive milestones.
selected_milestones AS (
    SELECT project_id, asset_id, activity_name, milestone_status
    FROM {SILVER}.milestones
    WHERE (project_id = 'AUS-BANK-01' AND activity_id = 'A44018')
       OR (project_id = 'HOU-MIXED-02' AND activity_id = 'A33011')
       OR (project_id = 'HAR-SEISMIC-03' AND activity_id = 'A22007')
),
-- Source freshness is separate from the time this notebook runs.
source_times AS (
    SELECT extracted_at FROM {SILVER}.assets
    UNION ALL
    SELECT extracted_at FROM {SILVER}.purchase_orders
    UNION ALL
    SELECT extracted_at FROM {SILVER}.milestones
    UNION ALL
    SELECT extracted_at FROM {SILVER}.inspections
),
oldest_source AS (
    SELECT MIN(extracted_at) AS source_extracted_at FROM source_times
)
SELECT a.project_id, a.asset_id, a.project_name,
       a.canonical_asset_name AS asset_name,
       CASE WHEN s.supplier_names = '' THEN 'UNRESOLVED'
            ELSE s.supplier_names END AS supplier_name,
       m.activity_name AS current_milestone,
       m.milestone_status, o.purchase_order_status, i.inspection_status,
       o.committed_cost_cents, 'USD' AS currency_code,
       o.purchase_order_count, i.inspection_count,
       CASE WHEN i.inspection_status = 'PASS' AND s.supplier_needs_review = 0
            THEN 'CONTEXT_AVAILABLE' ELSE 'REVIEW_REQUIRED' END AS decision_readiness,
       f.source_extracted_at,
       :refresh_time AS last_successful_refresh,
       :run_id AS run_id
FROM {SILVER}.assets a
INNER JOIN orders_by_asset o
    ON a.project_id = o.project_id AND a.asset_id = o.asset_id
INNER JOIN suppliers_by_asset s
    ON a.project_id = s.project_id AND a.asset_id = s.asset_id
INNER JOIN inspections_by_asset i
    ON a.project_id = i.project_id AND a.asset_id = i.asset_id
INNER JOIN selected_milestones m
    ON a.project_id = m.project_id AND a.asset_id = m.asset_id
CROSS JOIN oldest_source f
""", args={"refresh_time": datetime.now(timezone.utc).isoformat(), "run_id": RUN_ID})

# Recreate this Gold Delta table and save the SQL result once.
spark.sql(f"DROP TABLE IF EXISTS {GOLD}.project_context")
context.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD}.project_context")
display(spark.table(f"{GOLD}.project_context"))


## Task 2: Build milestone features

Read the persisted Gold project context and saved Silver tables. Select unfinished milestones due 1–30 days after the workshop's fixed reference date. Compute nine numeric inputs for Lab 2: days remaining, open-order fraction, inspection and supplier flags, milestone status flags, and log committed cost.


In [ ]:
FEATURES = ["days_to_planned_date", "open_po_fraction", "inspection_fail_flag",
    "inspection_info_flag", "supplier_review_flag", "unresolved_supplier_flag",
    "milestone_not_started_flag", "milestone_at_risk_flag", "log_committed_cost"]
context_input = spark.table(f"{GOLD}.project_context")
milestone_input = spark.table(f"{SILVER}.milestones")
order_input = spark.table(f"{SILVER}.purchase_orders")
supplier_input = spark.table(f"{SILVER}.suppliers")
as_of = config["fixture_as_of"]
eligible = milestone_input.withColumn("days_to_planned_date",
    F.datediff(F.to_date("planned_date"), F.to_date(F.lit(as_of[:10])))).filter(
        (F.coalesce(F.col("actual_date"), F.lit("")) == "") &
        (F.col("milestone_status") != "COMPLETE") & F.col("days_to_planned_date").between(1, 30))
po_features = order_input.groupBy(*keys).agg(
    F.avg((F.col("po_status") != "RECEIVED").cast("double")).alias("open_po_fraction"),
    F.max(F.col("supplier_id").isNull().cast("int")).alias("unresolved_supplier_flag"))
supplier_features = order_input.select(*keys, "supplier_id").distinct().join(
    supplier_input, "supplier_id", "left").groupBy(*keys).agg(
        F.max(F.when(F.col("supplier_id").isNull() | (F.col("qualification_status") != "APPROVED") |
            (F.col("compliance_status") != "EVIDENCE_PRESENT"), 1).otherwise(0)).alias("supplier_review_flag"))
features = (eligible.select(*keys, F.col("activity_id").alias("milestone_id"),
    F.col("activity_name").alias("milestone_name"), "planned_date", "days_to_planned_date",
    F.col("milestone_status").alias("upcoming_status"))
    .join(context_input.select(*keys, "project_name", "inspection_status", "committed_cost_cents", "source_extracted_at"), keys)
    .join(po_features, keys, "left").join(supplier_features, keys, "left")
    .select(*keys, "milestone_id", "project_name", "milestone_name", "planned_date",
        F.lit(as_of).alias("prediction_as_of"), "days_to_planned_date",
        F.coalesce(F.col("open_po_fraction"), F.lit(1.0)).alias("open_po_fraction"),
        (F.col("inspection_status") == "FAIL").cast("double").alias("inspection_fail_flag"),
        (F.col("inspection_status") == "REQUEST_INFO").cast("double").alias("inspection_info_flag"),
        F.coalesce(F.col("supplier_review_flag"), F.lit(0)).alias("supplier_review_flag"),
        F.coalesce(F.col("unresolved_supplier_flag"), F.lit(0)).alias("unresolved_supplier_flag"),
        (F.col("upcoming_status") == "NOT_STARTED").cast("double").alias("milestone_not_started_flag"),
        (F.col("upcoming_status") == "AT_RISK").cast("double").alias("milestone_at_risk_flag"),
        F.log1p(F.col("committed_cost_cents") / 100.0).alias("log_committed_cost"),
        "committed_cost_cents", "source_extracted_at", F.lit("PROVIDED_HISTORICAL_FIXTURE").alias("data_origin"),
        F.lit(datetime.now(timezone.utc).isoformat()).alias("last_successful_refresh"), F.lit(RUN_ID).alias("run_id")))
# Give all nine Lab 2 model inputs the same numeric type.
features = features.select(*[
    F.col(c).cast("double").alias(c) if c in FEATURES else F.col(c)
    for c in features.columns
])
# Recreate this Gold Delta table, then persist the complete Spark transformation once.
spark.sql(f"DROP TABLE IF EXISTS {GOLD}.milestone_features")
features.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD}.milestone_features")
display(spark.table(f"{GOLD}.milestone_features").select("project_id", "milestone_id", "planned_date", *FEATURES))


## Task 3: Inspect lineage

Open `seer_livelabs_20260922.seer_gold.milestone_features` in the Master catalog, then **Actions → Lineage (Preview)**. Inspect its `project_context` input and the Silver tables, then expand Silver inputs to Bronze. Workflow tasks now identify the Bronze, Silver, and Gold notebooks separately.

Business writes preserve native Spark source plans. Refresh the catalog and reopen current table identities after recreation. Lab 1 creates five Bronze, five Silver, and two Gold tables. Lab 2 adds only `milestone_delay_predictions`.
